In [1]:
import os
import pandas as pd
import psycopg2

DB_URL = os.environ.get(
    "DATABASE_URL",
    "postgresql://neondb_owner:npg_X71kIZjKPMcL"
    "@ep-twilight-cloud-agxexq52.c-2.eu-central-1.aws.neon.tech/DSPRO?sslmode=require"
)

BASE_CTE = """
WITH base AS (
    SELECT
        l.id AS listing_id,
        ld.address,
        ld.area_sqm,
        ld.rooms,
        sd.population,
        sd.oev_score,
        sd.solar_class,
        sd.elevation_m,
        sd.lv95_east,
        sd.lv95_north,
        lb.egid,
        gb.gbauj,
        gb.ganzwhg,
        gb.garea
    FROM listings l
    LEFT JOIN listing_details ld
        ON ld.listing_id = l.id
    LEFT JOIN swisstopo_details sd
        ON sd.listing_id = l.id
    LEFT JOIN listing_buildings lb
        ON lb.listing_id = l.id
    LEFT JOIN gwr_buildings gb
        ON gb.egid = lb.egid
)
"""

CUMULATIVE_FILTER_COUNTS_QUERY = BASE_CTE + """
SELECT '01_all_listings' AS step, COUNT(DISTINCT listing_id) AS records FROM base

UNION ALL
SELECT '02_address_not_null', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL

UNION ALL
SELECT '03_area_sqm', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL

UNION ALL
SELECT '04_population', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND population IS NOT NULL

UNION ALL
SELECT '05_egid', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL

UNION ALL
SELECT '06_gbauj', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL

UNION ALL
SELECT '07_ganzwhg', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL AND ganzwhg IS NOT NULL

UNION ALL
SELECT '08_garea', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL AND ganzwhg IS NOT NULL AND garea IS NOT NULL

UNION ALL
SELECT '09_oev_score', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL AND ganzwhg IS NOT NULL AND garea IS NOT NULL AND oev_score IS NOT NULL

UNION ALL
SELECT '10_solar_class', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL AND ganzwhg IS NOT NULL AND garea IS NOT NULL AND oev_score IS NOT NULL AND solar_class IS NOT NULL

UNION ALL
SELECT '11_elevation', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL AND ganzwhg IS NOT NULL AND garea IS NOT NULL AND oev_score IS NOT NULL AND solar_class IS NOT NULL AND elevation_m IS NOT NULL

ORDER BY step;
"""

FINAL_DATASET_QUERY = """
SELECT DISTINCT
    l.id AS listing_id,
    ld.address,
    ld.area_sqm,
    ld.rooms,
    sd.population,
    sd.oev_score,
    sd.solar_class,
    sd.elevation_m,
    sd.lv95_east,
    sd.lv95_north,
    lb.egid,
    gb.gbauj,
    gb.ganzwhg,
    gb.garea
FROM listings l
JOIN listing_details ld ON ld.listing_id = l.id
JOIN swisstopo_details sd ON sd.listing_id = l.id
JOIN listing_buildings lb ON lb.listing_id = l.id
JOIN gwr_buildings gb ON gb.egid = lb.egid
WHERE ld.address IS NOT NULL
  AND ld.area_sqm IS NOT NULL
  AND sd.population IS NOT NULL
  AND lb.egid IS NOT NULL
  AND gb.gbauj IS NOT NULL
  AND gb.ganzwhg IS NOT NULL
  AND gb.garea IS NOT NULL
  AND sd.oev_score IS NOT NULL
  AND sd.solar_class IS NOT NULL
  AND sd.elevation_m IS NOT NULL
ORDER BY listing_id;
"""

STATUS_QUERY = BASE_CTE + """
SELECT
    listing_id,
    CASE
        WHEN address IS NULL THEN 'missing_address'
        WHEN area_sqm IS NULL THEN 'missing_area_sqm'
        WHEN population IS NULL THEN 'missing_population'
        WHEN egid IS NULL THEN 'missing_egid'
        WHEN gbauj IS NULL THEN 'missing_gbauj'
        WHEN ganzwhg IS NULL THEN 'missing_ganzwhg'
        WHEN garea IS NULL THEN 'missing_garea'
        WHEN oev_score IS NULL THEN 'missing_oev_score'
        WHEN solar_class IS NULL THEN 'missing_solar_class'
        WHEN elevation_m IS NULL THEN 'missing_elevation'
        ELSE 'usable'
    END AS status
FROM base
ORDER BY listing_id;
"""

def q(conn, sql):
    return pd.read_sql_query(sql, conn)

conn = psycopg2.connect(DB_URL)

try:
    df_cumulative = q(conn, CUMULATIVE_FILTER_COUNTS_QUERY)
    df_final = q(conn, FINAL_DATASET_QUERY)
    df_status = q(conn, STATUS_QUERY)

    print("=== Kumulative Filter ===")
    display(df_cumulative)

    print("=== Final brauchbare Listings ===")
    display(df_final)

    print("=== Status je Listing ===")
    display(df_status)

    print("Final brauchbare Listings:", df_final["listing_id"].nunique())
    import os

    OUTPUT_DIR = "output_csv"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df_cumulative.to_csv(f"{OUTPUT_DIR}/cumulative_filters.csv", index=False)
    df_final.to_csv(f"{OUTPUT_DIR}/final_listings.csv", index=False)
    df_status.to_csv(f"{OUTPUT_DIR}/listing_status.csv", index=False)

    print(f"CSV Dateien gespeichert in: {OUTPUT_DIR}/")

finally:
    conn.close()

/tmp/ipykernel_4455/1289709468.py:141: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


=== Kumulative Filter ===


,step,records
0,01_all_listings,9994
1,02_address_not_null,9839
2,03_area_sqm,9608
3,04_population,8921
4,05_egid,8920
5,06_gbauj,5717
6,07_ganzwhg,5103
7,08_garea,5100
8,09_oev_score,5099
9,10_solar_class,4547


=== Final brauchbare Listings ===


,listing_id,address,area_sqm,rooms,population,oev_score,solar_class,elevation_m,lv95_east,lv95_north,egid,gbauj,ganzwhg,garea
0,1,"8044, Gockhausen-Zürich, Rütistrasse, 1",34,2.0,44,2043.0,2.0,572.9,2687446.25,1248599.000,93458,1971,1,192.0
1,2,"3904, Naters, Bahnhofstrasse, 8",108,4.0,107,5656.0,3.0,672.3,2642156.00,1130267.625,892059,1974,34,2000.0
2,4,"2540, Grenchen, Güterstrasse, 17",97,4.0,184,12617.0,4.0,439.6,2597233.25,1226600.375,191354719,2017,18,367.0
3,5,"1030, Bussigny, Rue de Lausanne 52 D, 52D",23,1.0,129,18439.0,4.0,430.4,2532786.25,1155625.250,794334,1948,50,731.0
4,6,"2732, Reconvilier, Route De Chaindon, 8",76,3.0,48,717.0,3.0,743.6,2583429.75,1231621.000,191362735,2018,17,587.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4542,9984,"7000, Chur, Pulvermühlestrasse, 13",103,4.0,142,7292.0,3.0,579.6,2758484.25,1191052.625,1739419,1972,36,517.0
4543,9992,"Flurstrasse **, Kloten, Flurstrasse 11",105,2.0,134,9175.0,2.0,445.9,2686898.00,1256475.875,21844,1962,11,240.0
4544,9994,"6900, Massagno, Via Sione, 10",74,3.0,105,3750.0,4.0,380.3,2716700.75,1096886.875,11143819,1967,20,328.0
4545,9995,"4057, Basel, Offenburgerstrasse, 35",65,3.0,219,17569.0,5.0,251.2,2611255.00,1268742.625,457153,1899,5,93.0


=== Status je Listing ===


,listing_id,status
0,1,usable
1,2,usable
2,3,missing_gbauj
3,4,usable
4,5,usable
...,...,...
9989,9992,usable
9990,9993,missing_population
9991,9994,usable
9992,9995,usable


Final brauchbare Listings: 4547
CSV Dateien gespeichert in: output_csv/


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine

DB_URL = os.environ.get(
    "DATABASE_URL",
    "postgresql://neondb_owner:npg_X71kIZjKPMcL"
    "@ep-twilight-cloud-agxexq52.c-2.eu-central-1.aws.neon.tech/DSPRO?sslmode=require"
)

BASE_CTE = """
WITH base AS (
    SELECT
        l.id AS listing_id,
        ld.address,
        ld.area_sqm,
        ld.rooms,
        ld.price_cold,
        sd.population,
        sd.oev_score,
        sd.solar_class,
        sd.elevation_m,
        sd.lv95_east,
        sd.lv95_north,
        lb.egid,
        gb.gbauj,
        gb.ganzwhg,
        gb.garea
    FROM listings l
    LEFT JOIN listing_details ld
        ON ld.listing_id = l.id
    LEFT JOIN swisstopo_details sd
        ON sd.listing_id = l.id
    LEFT JOIN listing_buildings lb
        ON lb.listing_id = l.id
    LEFT JOIN gwr_buildings gb
        ON gb.egid = lb.egid
)
"""

CUMULATIVE_FILTER_COUNTS_QUERY = BASE_CTE + """
SELECT '01_all_listings' AS step, COUNT(DISTINCT listing_id) AS records FROM base

UNION ALL
SELECT '02_address_not_null', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL

UNION ALL
SELECT '03_area_sqm', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL

UNION ALL
SELECT '04_rooms', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND rooms IS NOT NULL

UNION ALL
SELECT '05_price_cold', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND rooms IS NOT NULL AND price_cold IS NOT NULL

UNION ALL
SELECT '06_population', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND rooms IS NOT NULL AND price_cold IS NOT NULL AND population IS NOT NULL

UNION ALL
SELECT '07_egid', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND rooms IS NOT NULL AND price_cold IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL

UNION ALL
SELECT '08_gbauj', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND rooms IS NOT NULL AND price_cold IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL

UNION ALL
SELECT '09_ganzwhg', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND rooms IS NOT NULL AND price_cold IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL AND ganzwhg IS NOT NULL

UNION ALL
SELECT '10_garea', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND rooms IS NOT NULL AND price_cold IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL AND ganzwhg IS NOT NULL AND garea IS NOT NULL

UNION ALL
SELECT '11_oev_score', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND rooms IS NOT NULL AND price_cold IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL AND ganzwhg IS NOT NULL AND garea IS NOT NULL AND oev_score IS NOT NULL

UNION ALL
SELECT '12_solar_class', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND rooms IS NOT NULL AND price_cold IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL AND ganzwhg IS NOT NULL AND garea IS NOT NULL AND oev_score IS NOT NULL AND solar_class IS NOT NULL

UNION ALL
SELECT '13_elevation', COUNT(DISTINCT listing_id)
FROM base WHERE address IS NOT NULL AND area_sqm IS NOT NULL AND rooms IS NOT NULL AND price_cold IS NOT NULL AND population IS NOT NULL AND egid IS NOT NULL AND gbauj IS NOT NULL AND ganzwhg IS NOT NULL AND garea IS NOT NULL AND oev_score IS NOT NULL AND solar_class IS NOT NULL AND elevation_m IS NOT NULL

ORDER BY step;
"""

FINAL_DATASET_QUERY = """
SELECT DISTINCT
    l.id AS listing_id,
    ld.address,
    ld.area_sqm,
    ld.rooms,
    ld.price_cold,
    sd.population,
    sd.oev_score,
    sd.solar_class,
    sd.elevation_m,
    sd.lv95_east,
    sd.lv95_north,
    lb.egid,
    gb.gbauj,
    gb.ganzwhg,
    gb.garea
FROM listings l
JOIN listing_details ld ON ld.listing_id = l.id
JOIN swisstopo_details sd ON sd.listing_id = l.id
JOIN listing_buildings lb ON lb.listing_id = l.id
JOIN gwr_buildings gb ON gb.egid = lb.egid
WHERE ld.address IS NOT NULL
  AND ld.area_sqm IS NOT NULL
  AND ld.rooms IS NOT NULL
  AND ld.price_cold IS NOT NULL
  AND sd.population IS NOT NULL
  AND lb.egid IS NOT NULL
  AND gb.gbauj IS NOT NULL
  AND gb.ganzwhg IS NOT NULL
  AND gb.garea IS NOT NULL
  AND sd.oev_score IS NOT NULL
  AND sd.solar_class IS NOT NULL
  AND sd.elevation_m IS NOT NULL
ORDER BY listing_id;
"""

STATUS_QUERY = BASE_CTE + """
SELECT
    listing_id,
    CASE
        WHEN address IS NULL THEN 'missing_address'
        WHEN area_sqm IS NULL THEN 'missing_area_sqm'
        WHEN rooms IS NULL THEN 'missing_rooms'
        WHEN price_cold IS NULL THEN 'missing_price_cold'
        WHEN population IS NULL THEN 'missing_population'
        WHEN egid IS NULL THEN 'missing_egid'
        WHEN gbauj IS NULL THEN 'missing_gbauj'
        WHEN ganzwhg IS NULL THEN 'missing_ganzwhg'
        WHEN garea IS NULL THEN 'missing_garea'
        WHEN oev_score IS NULL THEN 'missing_oev_score'
        WHEN solar_class IS NULL THEN 'missing_solar_class'
        WHEN elevation_m IS NULL THEN 'missing_elevation'
        ELSE 'usable'
    END AS status
FROM base
ORDER BY listing_id;
"""

def q(conn, sql):
    return pd.read_sql_query(sql, conn)

engine = create_engine(DB_URL)

try:
    with engine.connect() as conn:
        df_cumulative = q(conn, CUMULATIVE_FILTER_COUNTS_QUERY)
        df_final = q(conn, FINAL_DATASET_QUERY)
        df_status = q(conn, STATUS_QUERY)

        print("=== Kumulative Filter ===")
        display(df_cumulative)

        print("=== Final brauchbare Listings ===")
        display(df_final)

        print("=== Status je Listing ===")
        display(df_status)

        print("Final brauchbare Listings:", df_final["listing_id"].nunique())

        # Create model dataset with only ML features
        model_features = [
            'area_sqm', 'rooms', 'price_cold', 'population', 'oev_score',
            'solar_class', 'elevation_m', 'lv95_east', 'lv95_north',
            'gbauj', 'ganzwhg', 'garea'
        ]
        df_model = df_final[model_features].copy()

        import os

        OUTPUT_DIR = "output_csv"
        os.makedirs(OUTPUT_DIR, exist_ok=True)

        df_cumulative.to_csv(f"{OUTPUT_DIR}/cumulative_filters.csv", index=False)
        df_final.to_csv(f"{OUTPUT_DIR}/final_listings.csv", index=False)
        df_status.to_csv(f"{OUTPUT_DIR}/listing_status.csv", index=False)
        df_model.to_csv(f"{OUTPUT_DIR}/model.csv", index=False)

        print(f"CSV Dateien gespeichert in: {OUTPUT_DIR}/")
        print(f"  - cumulative_filters.csv ({len(df_cumulative)} rows)")
        print(f"  - final_listings.csv ({len(df_final)} rows)")
        print(f"  - listing_status.csv ({len(df_status)} rows)")
        print(f"  - model.csv ({len(df_model)} rows, {len(model_features)} features)")

finally:
    engine.dispose()

=== Kumulative Filter ===


,step,records
0,01_all_listings,9994
1,02_address_not_null,9839
2,03_area_sqm,9608
3,04_rooms,9586
4,05_price_cold,9582
5,06_population,8897
6,07_egid,8896
7,08_gbauj,5706
8,09_ganzwhg,5092
9,10_garea,5089


=== Final brauchbare Listings ===


,listing_id,address,area_sqm,rooms,price_cold,population,oev_score,solar_class,elevation_m,lv95_east,lv95_north,egid,gbauj,ganzwhg,garea
0,1,"8044, Gockhausen-Zürich, Rütistrasse, 1",34,2,850,44,2043.0,2.0,572.9,2687446.25,1248599.000,93458,1971,1,192.0
1,2,"3904, Naters, Bahnhofstrasse, 8",108,4,1990,107,5656.0,3.0,672.3,2642156.00,1130267.625,892059,1974,34,2000.0
2,4,"2540, Grenchen, Güterstrasse, 17",97,4,1520,184,12617.0,4.0,439.6,2597233.25,1226600.375,191354719,2017,18,367.0
3,5,"1030, Bussigny, Rue de Lausanne 52 D, 52D",23,1,1396,129,18439.0,4.0,430.4,2532786.25,1155625.250,794334,1948,50,731.0
4,6,"2732, Reconvilier, Route De Chaindon, 8",76,3,1400,48,717.0,3.0,743.6,2583429.75,1231621.000,191362735,2018,17,587.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4531,9984,"7000, Chur, Pulvermühlestrasse, 13",103,4,2090,142,7292.0,3.0,579.6,2758484.25,1191052.625,1739419,1972,36,517.0
4532,9992,"Flurstrasse **, Kloten, Flurstrasse 11",105,2,3660,134,9175.0,2.0,445.9,2686898.00,1256475.875,21844,1962,11,240.0
4533,9994,"6900, Massagno, Via Sione, 10",74,3,1380,105,3750.0,4.0,380.3,2716700.75,1096886.875,11143819,1967,20,328.0
4534,9995,"4057, Basel, Offenburgerstrasse, 35",65,3,1266,219,17569.0,5.0,251.2,2611255.00,1268742.625,457153,1899,5,93.0


=== Status je Listing ===


,listing_id,status
0,1,usable
1,2,usable
2,3,missing_gbauj
3,4,usable
4,5,usable
...,...,...
9989,9992,usable
9990,9993,missing_population
9991,9994,usable
9992,9995,usable


Final brauchbare Listings: 4536
CSV Dateien gespeichert in: output_csv/
  - cumulative_filters.csv (13 rows)
  - final_listings.csv (4536 rows)
  - listing_status.csv (9994 rows)
  - model.csv (4536 rows, 12 features)
